# Rekaman Prompt Hari 2 — Data Cleaning & EDA dengan AI

Pelatihan Machine Learning untuk Operasi PLTP · Hari 2, 10.00 · lembar kerja di akhir deck `materi-3-ai`

> **Rencana C — rekaman prompt.** Notebook ini dipakai bila AI coding agent tidak bisa
> diakses dari tempat pelatihan. Setiap langkah berisi *prompt* (sama dengan di slide) dan
> *kode yang dihasilkan AI* untuk prompt itu. Tugas Anda tetap sama dengan vibe coding:
> **baca kodenya, tebak hasilnya, jalankan, lalu periksa.**
>
> Begitu akses AI tersedia (VPN, hotspot, atau AI lokal perusahaan), ketik prompt yang sama
> ke agent dan bandingkan kodenya dengan rekaman ini. Tidak harus sama persis — yang harus
> sama adalah *angkanya*.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

df = pd.read_csv("../data/produksi_bulanan.csv")
print("Awal:", len(df), "baris,", df["kode_unit"].nunique(), "kode unit")

### Langkah 1 — temukan (jangan perbaiki dulu)

> **Prompt:** Periksa `data/produksi_bulanan.csv`: duplikat, nilai kosong, variasi penulisan `kode_unit`, nilai di luar rentang wajar `data/kamus_data.md`. Tampilkan dalam satu tabel ringkasan. Jangan ubah apa pun.

**Tebak dulu sebelum menjalankan sel di bawah:** berapa masalah yang akan ditemukan?

In [ ]:
rentang = {  # dari data/kamus_data.md
    "cf_persen": (0, 95), "jam_operasi": (0, 744),
    "energi_aktual_mwh": (0, 75000), "uap_aktual_ton": (0, 700000),
}
temuan = [
    ("baris duplikat", int(df.duplicated().sum())),
    ("nilai kosong (semua kolom)", int(df.isna().sum().sum())),
    ("kode_unit bukan huruf besar", int((df["kode_unit"] != df["kode_unit"].str.upper()).sum())),
]
for kolom, (lo, hi) in rentang.items():
    temuan.append((f"{kolom} di luar {lo}–{hi}", int((~df[kolom].between(lo, hi) & df[kolom].notna()).sum())))

display(pd.DataFrame(temuan, columns=["pemeriksaan", "jumlah"]))

**Periksa sebelum percaya:**

- [ ] 1 duplikat, 12 kosong, 3 kode unit salah ketik, 0 nilai di luar rentang.
- [ ] File CSV asli tidak berubah (kode hanya membaca).

### Langkah 2a — seragamkan kode unit

> **Prompt:** Seragamkan `kode_unit` ke huruf besar. Cetak jumlah unit unik sebelum dan sesudah.

**Tebak dulu sebelum menjalankan sel di bawah:** 9 → berapa?

In [ ]:
print("Sebelum:", df["kode_unit"].nunique(), "unit")
df["kode_unit"] = df["kode_unit"].str.upper()
print("Sesudah:", df["kode_unit"].nunique(), "unit")

### Langkah 2b — duplikat

> **Prompt:** Tampilkan baris duplikat. Setelah saya setujui, hapus dan cetak jumlah baris.

**Tebak dulu sebelum menjalankan sel di bawah:** baris mana yang ganda?

In [ ]:
display(df[df.duplicated(keep=False)])   # tampilkan kedua salinan

In [ ]:
# Dijalankan SETELAH Anda setuju
sebelum = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Baris: {sebelum} -> {len(df)}")

### Langkah 3 — nilai kosong adalah keputusan

> **Prompt:** Untuk 12 baris dengan `energi_aktual_mwh` kosong, tampilkan nilai `cf_persen` dan `jam_operasi`-nya. Apakah unit tersebut sebenarnya beroperasi? Usulkan cara mengisi yang paling masuk akal dan jelaskan risikonya. Jangan isi dulu.

**Tebak dulu sebelum menjalankan sel di bawah:** apakah unit-unit itu beroperasi pada bulan tersebut?

In [ ]:
kosong = df[df["energi_aktual_mwh"].isna()]
display(kosong[["tanggal", "kode_unit", "cf_persen", "jam_operasi", "energi_rencana_mwh"]])

print("Rata-rata energi, kosong diabaikan:", round(df["energi_aktual_mwh"].mean()))
print("Rata-rata energi, kosong diisi 0   :", round(df["energi_aktual_mwh"].fillna(0).mean()))

**Usulan AI (contoh):** CF dan jam operasi tidak nol, jadi unit *beroperasi* — mengisi 0 akan salah.
Energi bisa dihitung ulang dari definisi CF: `energi = cf/100 × kapasitas_mw × jam_dalam_bulan`.
Risikonya: CF sendiri mungkin dihitung dari energi, sehingga nilai isian ini *perkiraan*, bukan pengukuran.
Tandai baris yang diisi supaya bisa dilacak.

**Keputusan Anda** (tulis di prompt log), lalu jalankan sel berikut bila setuju:

In [ ]:
unit = pd.read_csv("../data/unit_pembangkit.csv")[["kode_unit", "kapasitas_mw"]]
df = df.merge(unit, on="kode_unit", how="left")

df["tanggal"] = pd.to_datetime(df["tanggal"])
jam_bulan = df["tanggal"].dt.days_in_month * 24

df["energi_diisi"] = df["energi_aktual_mwh"].isna()           # penanda
perkiraan = df["cf_persen"] / 100 * df["kapasitas_mw"] * jam_bulan
df["energi_aktual_mwh"] = df["energi_aktual_mwh"].fillna(perkiraan.round(0))

print("Nilai kosong tersisa:", df["energi_aktual_mwh"].isna().sum())
print("Baris yang diisi:", df["energi_diisi"].sum())

**Periksa sebelum percaya:**

- [ ] Rumus CF dari kamus data: `energi ÷ (kapasitas × jam_dalam_bulan) × 100`.
- [ ] Bandingkan satu baris yang diisi dengan `energi_rencana_mwh`-nya — masuk akal?
- [ ] `merge` dengan `how="left"` tidak menambah atau mengurangi baris (cek `len(df)`).

### Langkah 4 — EDA lanjutan pada data bersih

> **Prompt:** Dengan data bersih, ulangi boxplot CF per unit. Lalu gabungkan dengan `unit_pembangkit.csv`: apakah CF berbeda menurut `jenis_turbin`?

**Tebak dulu sebelum menjalankan sel di bawah:** sekarang berapa kotak? Turbin jenis apa CF-nya tertinggi?

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
df.boxplot(column="cf_persen", by="kode_unit", ax=ax)
plt.suptitle(""); ax.set_title("CF per unit — data bersih"); ax.set_ylabel("CF (%)")
plt.show()

turbin = pd.read_csv("../data/unit_pembangkit.csv")[["kode_unit", "jenis_turbin"]]
print(df.merge(turbin, on="kode_unit").groupby("jenis_turbin")["cf_persen"].agg(["mean", "median", "count"]).round(1))

### Langkah 5 — simpan dengan pengaman

> **Prompt:** Simpan ke `data/olahan/produksi_bersih.csv`. Tambahkan `assert`: 1.080 baris, 6 unit, tidak ada duplikat. Lalu Restart & Run All.

**Tebak dulu sebelum menjalankan sel di bawah:** apakah semua assert lolos?

In [ ]:
assert len(df) == 1080, f"jumlah baris {len(df)}, seharusnya 1080"
assert df["kode_unit"].nunique() == 6, "jumlah unit bukan 6"
assert not df.duplicated(subset=["tanggal", "kode_unit"]).any(), "masih ada unit-bulan ganda"
assert df["energi_aktual_mwh"].notna().all(), "masih ada energi kosong"

Path("../data/olahan").mkdir(exist_ok=True)
df.drop(columns=["kapasitas_mw"]).to_csv("../data/olahan/produksi_bersih.csv", index=False)
print("Tersimpan: data/olahan/produksi_bersih.csv —", len(df), "baris, 6 unit")

**Periksa sebelum percaya:**

- [ ] Restart & Run All berhasil tanpa error.
- [ ] Angka sama dengan pasangan sebelah: 1.080 baris, 6 unit, 12 nilai diisi.
- [ ] File asli `data/produksi_bulanan.csv` tetap 1.081 baris.